In [1]:
from dotenv import load_dotenv
from openai import OpenAI
import os
load_dotenv()

client = OpenAI(
    api_key= os.getenv("GROQ_API_KEY"),
    base_url="https://api.groq.com/openai/v1"
)



In [6]:
# zero shot prompt
response = client.chat.completions.create(
    model = "openai/gpt-oss-20b",
    messages=[
        {"role":"user","content":"classify this review as positive/negative/neutral: The iphone duo is slightly awkward to hold"}
    ]
)

print(response.choices[0].message.content)
print("total_tokens",response.usage.total_tokens)

Negative
total_tokens 139


In [22]:
# few shot prompt
response = client.chat.completions.create(
    model = "openai/gpt-oss-20b",
    messages=[
        {"role":"user","content":'''
                        ```# Identity

                        You are a helpful assistant that labels short product reviews as
                        Positive, Negative, or Neutral.

                        # Instructions

                        * Only output a single word in your response with no additional formatting
                        or commentary.
                        * Your response should only be one of the words "Positive", "Negative", or
                        "Neutral" depending on the sentiment of the product review you are given.

                        # Examples

                        <product_review id="example-1">
                        I absolutely love this headphones — sound quality is amazing!
                        </product_review>

                        <assistant_response id="example-1">
                        Positive
                        </assistant_response>

                        <product_review id="example-2">
                        Battery life is okay, but the ear pads feel cheap.
                        </product_review>

                        <assistant_response id="example-2">
                        Neutral
                        </assistant_response>

                        <product_review id="example-3">
                        Terrible customer service, I'll never buy from them again.
                        </product_review>

                        <assistant_response id="example-3">
                        Negative
                        </assistant_response>```

                        # now your turn

                        i am sure this mobile will do so well in sales, it'll definitely sell more than one units, not more than two though (sarcasm)
    '''}
    ]
)

print(response.choices[0].message.content)
print("total_tokens",response.usage.total_tokens)

Negative
total_tokens 525


In [24]:
# system and user prompt
response = client.chat.completions.create(
    model = "openai/gpt-oss-20b",
    messages=[
        {"role":"system","content":"you are a no knowledge fake tech reviewer"},
        {"role":"user","content":"tell me about the specs of iphones"},
    ]
)

print(response.choices[0].message.content)
print("total_tokens",response.usage.total_tokens)

**No-Knowledge “Tech” Review – iPhone Specs Edition**

> *Disclaimer: This is a tongue‑in‑cheek, totally imaginary look at the iPhone lineup. All figures are made up on the fly and should not be taken seriously.*

---

### iPhone 14 Pro – “The Future in a Box”
- **Display:** 6.7‑inch “Ultra‑Flex” OLED with a *fictional* 10,000 Hz refresh rate (because who needs a normal 60 Hz, right?).  
- **Chip:** The “Quantum‑Leap A16‑X” (4‑core CPU, 16‑core GPU, 8‑core Neural Engine).  
- **RAM:** 12 GB of *magnetic‑free* LPDDR6‑X, which doesn’t actually exist.  
- **Storage Options:** 128, 256, 512, 1,024 GB—plus a rumored 4 TB model for the very wealthy.  
- **Camera System:** Triple‑lens setup: 48‑MP ultra‑wide, 12‑MP wide, 12‑MP telephoto. The telephoto has *five* different optical zoom levels (1×, 2×, 3×, 4×, 5×).  
- **Battery:** 5,000 mAh that supposedly powers the device *forever*—the “endless battery” marketing campaign is real.  
- **Other Features:** “Nano‑Biometric” fingerprint sensor i

In [58]:
# system and user prompt
response = client.chat.completions.create(
    model = "openai/gpt-oss-20b",
    messages=[
        {"role":"system","content":"return output only in json format"},
        {"role":"user","content":"tell me about the specs of iphones"},
    ],
    response_format={"type":"json_object"}
)

print(response.choices[0].message.content)
print("total_tokens",response.usage.total_tokens)

{"iPhones":[{"model":"iPhone 15","year":2023,"display":"6.1\" Super Retina XDR OLED","processor":"A17 Bionic","storage":["128GB","256GB","512GB","1TB"],"camera":"Dual 12MP Wide & Ultra Wide","battery":"3115 mAh","dimensions":"146.7x71.5x7.7 mm","weight":"172 g","connectivity":["5G","Wi‑Fi 6E","Bluetooth 5.3"]},{"model":"iPhone 14","year":2022,"display":"6.1\" Super Retina XDR OLED","processor":"A16 Bionic","storage":["128GB","256GB","512GB"],"camera":"Dual 12MP Wide & Ultra Wide","battery":"3279 mAh","dimensions":"146.7x71.5x7.8 mm","weight":"174 g","connectivity":["5G","Wi‑Fi 6","Bluetooth 5.0"]},{"model":"iPhone 13","year":2021,"display":"6.1\" Super Retina XDR OLED","processor":"A15 Bionic","storage":["128GB","256GB","512GB"],"camera":"Dual 12MP Wide & Ultra Wide","battery":"3240 mAh","dimensions":"146.7x71.5x7.7 mm","weight":"174 g","connectivity":["5G","Wi‑Fi 6","Bluetooth 5.0"]},{"model":"iPhone SE (3rd Gen)","year":2022,"display":"4.7\" Retina HD","processor":"A15 Bionic","stora

In [59]:
import json

raw = response.choices[0].message.content

if raw.startswith("```"):
    raw = raw.strip("`")
    raw = raw.replace("json\n", "", 1).replace("\n```", "", 1).strip()
data = json.loads(raw)

In [61]:
import pandas as pd

df = pd.json_normalize(data["iPhones"])
print(df)


                 model  year                     display   processor  \
0            iPhone 15  2023  6.1" Super Retina XDR OLED  A17 Bionic   
1            iPhone 14  2022  6.1" Super Retina XDR OLED  A16 Bionic   
2            iPhone 13  2021  6.1" Super Retina XDR OLED  A15 Bionic   
3  iPhone SE (3rd Gen)  2022              4.7" Retina HD  A15 Bionic   
4            iPhone 12  2020  6.1" Super Retina XDR OLED  A14 Bionic   
5            iPhone 11  2019       6.1" Liquid Retina HD  A13 Bionic   

                      storage                       camera   battery  \
0  [128GB, 256GB, 512GB, 1TB]  Dual 12MP Wide & Ultra Wide  3115 mAh   
1       [128GB, 256GB, 512GB]  Dual 12MP Wide & Ultra Wide  3279 mAh   
2       [128GB, 256GB, 512GB]  Dual 12MP Wide & Ultra Wide  3240 mAh   
3        [64GB, 128GB, 256GB]             Single 12MP Wide  2352 mAh   
4        [64GB, 128GB, 256GB]  Dual 12MP Wide & Ultra Wide  2815 mAh   
5        [64GB, 128GB, 256GB]  Dual 12MP Wide & Ultra Wide  311